# 12 - XGBoost Model: T-cell and TMB

**Purpose**: Compares XGBoost vs logistic regression for predicting binary ICI response.
Predictors: `log2(CIDER T-cell Score + 1)` and `log2(TMB per Mb + 1)`.

Evaluation is two-stage to avoid tuning bias:
1. **Tune**: `xgb.cv` with early stopping selects hyperparameters on the full training data.
2. **Evaluate**: the same 10-fold × 10-repeat CV used for logistic regression is run with
   those fixed hyperparameters (single-row `tuneGrid` — no selection within folds).

| | |
|---|---|
| **Inputs**  | `CIDER_PROJ_DIR/cpi_ready.rds` |
| **Outputs** | `figs/supplement_xgboost_roc.png` |

In [1]:
source(paste0(dirname(getwd()), "/helpers/map.r"))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))
source(paste0(dirname(getwd()), "/helpers/config.r"))
source(paste0(dirname(getwd()), "/helpers/themes.r"))

dir.create(paste0(CIDER_PROJ_DIR, "figs/"), showWarnings = FALSE, recursive = TRUE)

In [2]:
library(pROC)
library(caret)
library(xgboost)
library(patchwork)

Type 'citation("pROC")' for a citation.


Attaching package: ‘pROC’


The following objects are masked from ‘package:stats’:

    cov, smooth, var


Loading required package: lattice


Attaching package: ‘caret’


The following object is masked from ‘package:purrr’:

    lift



Attaching package: ‘xgboost’


The following object is masked from ‘package:dplyr’:

    slice




# 0 - Load data

In [ ]:
# Same data preparation as 9_roc.ipynb — T-cell group, documented response only.
# cider_log2 = log2(cider_score_depth_adj + 1)
# tmb_log2   = log2(tmbPerMb + 1)  — tmbPerMb is the raw value here (not
# pre-transformed upstream -- see 0_prepare/3_meta.ipynb); log2 is applied
# at this point of use instead, same convention as CIDER.
roc_data <- readRDS(paste0(CIDER_PROJ_DIR, "cpi_ready.rds")) |>
  dplyr::filter(gp == "T-cell", !is.na(response)) |>
  dplyr::select(sampleId, cohort_broad, cohort_broad_go, response,
                cider_score_depth_adj, tmbPerMb, msi_high) |>
  dplyr::mutate(
    response   = as.integer(response),
    cider_log2 = log2(cider_score_depth_adj + 1),
    tmb_log2   = log2(tmbPerMb + 1)
  )

cv_data <- roc_data |>
  dplyr::filter(!is.na(response), !is.na(cider_log2), !is.na(tmb_log2)) |>
  dplyr::mutate(
    response_fct = factor(ifelse(response == 1, "Responder", "NonResponder"),
                          levels = c("Responder", "NonResponder"))
  )

message("Samples for modelling: ", nrow(cv_data))

# -- Merge MSI-high cohorts for the cohort-level AUC breakdown ----------------
# Per explicit request: rather than treating the raw "Other" bucket and the
# existing "CRC-MSI" cohort as separate facets below, pull the MSI-high half
# out of "Other" (same msi_high flag used project-wide, e.g.
# helpers/shortcuts.r::split_other_by_msi()) and pool it with "CRC-MSI" into
# one combined "ICI-MSI" cohort (MSI-high patients across every tumour
# type); the MSI-stable remainder of "Other" is relabelled "Other MSS" so
# every non-MSI cohort in this figure is explicitly MSS. This is a
# figure-local scheme, distinct from BROAD_LEVELS_OTHER_SPLIT (which keeps
# Non-CRC MSI and CRC-MSI as two separate cohorts in Figures 3/4) -- here
# they're pooled since this panel reports model AUC, not a response-rate
# comparison between colorectal and non-colorectal MSI tumours. Done here,
# before cohort_broad is dummy-encoded (next cell) and used as a factor in
# the Full model's own formula, so every downstream use (model features,
# per-cohort AUC) sees the same merged cohort definition.
cv_data <- cv_data |>
  dplyr::mutate(
    cohort_broad = dplyr::case_when(
      cohort_broad == "CRC-MSI"              ~ "ICI-MSI",
      cohort_broad == "Other" & msi_high == 1 ~ "ICI-MSI",
      cohort_broad == "Other"                 ~ "Other MSS",
      TRUE                                     ~ as.character(cohort_broad)
    )
  )

# Recompute cohort_broad_go (the "<cohort> (N = ...)" facet label) from this
# notebook's own final modelling population rather than reusing the
# upstream cpi_ready.rds label -- no pre-existing label exists yet for the
# newly merged ICI-MSI/Other MSS cohorts, and this way the N shown always
# matches what's actually plotted.
cohort_n <- cv_data |> dplyr::count(cohort_broad, name = "n_broad")
cv_data <- cv_data |>
  dplyr::select(-cohort_broad_go) |>
  dplyr::left_join(cohort_n, by = "cohort_broad") |>
  dplyr::mutate(cohort_broad_go = paste0(cohort_broad, " (N = ", n_broad, ")")) |>
  dplyr::select(-n_broad)

message("Cohorts after ICI-MSI merge: ", paste(sort(unique(cv_data$cohort_broad)), collapse = ", "))

In [9]:
extra_markers <- readRDS(paste0(CIDER_PROJ_DIR, "extra_markers.rds"))
# Filter out the pooled "ICI-Treated" rows: each patient appears once in a
# tumor-type cohort (cohort_broad) AND once in the pan-cancer pool.
# Keeping only tumor-type rows gives one row per patient with cohort annotation.
cv_data <- cv_data %>% inner_join(extra_markers) %>% filter(cohort_broad != "ICI-Treated")

Joining with `by = join_by(sampleId)`


In [ ]:
# Dummy-encode cohort_broad for XGBoost (xgbTree requires numeric input).
# model.matrix with -1 gives one column per level (no reference dropped),
# which is appropriate for tree-based models.
cohort_mat  <- model.matrix(~ cohort_broad - 1, cv_data)
cohort_cols <- colnames(cohort_mat)  # e.g. cohort_broadSKCM, cohort_broadNSCLC ...
cv_data     <- dplyr::bind_cols(cv_data, as.data.frame(cohort_mat))
message('Cohort dummy columns: ', paste(cohort_cols, collapse = ', '))

In [ ]:
extra_wgs_biomarkers <- c("cohort_broad", "msi_high", "hla_loh",
                          "driver_POLE", "driver_POLD1",
                          "driver_KEAP1", "driver_STK11",
                          "driver_EGFR", "driver_JAK1",
                          "driver_JAK2", "driver_B2M")

# Drop NAs across ALL features used in any model so that rowIndex is
# consistent across formula-based and matrix-based models.
# Without this, model.matrix silently drops NA rows and misaligns rowIndex.
all_features <- c("cider_log2", "tmb_log2", extra_wgs_biomarkers)
cv_data <- cv_data |> tidyr::drop_na(dplyr::all_of(all_features))
message("Samples after dropping NAs in all feature columns: ", nrow(cv_data))

# Recompute label_01 after NA drop
label_01 <- as.integer(cv_data$response_fct == "Responder")

# 1 - Tune XGBoost hyperparameters via xgb.cv

Tuning is completely separate from the evaluation CV.
The best `nrounds`, `max_depth`, and `eta` are selected here;
the evaluation step (section 2) uses them as fixed constants.

In [ ]:
# Tune XGBoost hyperparameters using xgb.cv with conservative, anti-overfitting defaults.
# Returns a single-row data.frame compatible with caret's tuneGrid.
tune_xgb_params <- function(X, y,
                            max_depth_vals = c(1),       # Decision stumps only
                            eta_vals       = c(0.01),    # Low learning rate
                            nrounds_max    = 300,       # Capped max iterations
                            nfold          = 5,
                            early_stop     = 10,        # Aggressive early stopping
                            seed           = 6222020) {
  set.seed(seed)
  dtrain   <- xgboost::xgb.DMatrix(data = as.matrix(X), label = y)
  best     <- NULL
  best_auc <- -Inf

  # Fixed conservative regularizers
  cons_gamma            <- .5  # High minimum loss reduction to split
  cons_min_child_weight <- 5   # Requires >=10 weighted samples per leaf
  cons_subsample        <- .8  # Bagging row fraction
  cons_colsample        <- 1  # Feature sampling fraction

  for (md in max_depth_vals) {
    for (eta in eta_vals) {
      params <- list(
        objective        = "binary:logistic",
        eval_metric      = "auc",
        max_depth        = md,
        eta              = eta,
        gamma            = cons_gamma,
        colsample_bytree = cons_colsample,
        min_child_weight = cons_min_child_weight,
        subsample        = cons_subsample
      )
      
      cv <- xgboost::xgb.cv(
        params                = params,
        data                  = dtrain,
        nrounds               = nrounds_max,
        nfold                 = nfold,
        early_stopping_rounds = early_stop,
        verbose               = FALSE
      )
      
      iter_auc <- cv$evaluation_log$test_auc_mean[cv$best_iteration]
      
      if (iter_auc > best_auc) {
        best_auc <- iter_auc
        # Matched exactly to the conservative params used during CV
        best <- data.frame(
          nrounds          = cv$best_iteration,
          max_depth        = md,
          eta              = eta,
          gamma            = cons_gamma,
          colsample_bytree = cons_colsample,
          min_child_weight = cons_min_child_weight,
          subsample        = cons_subsample
        )
      }
    }
  }
  
  message(sprintf("  max_depth=%d  eta=%.2f  nrounds=%d  (tune AUC=%.3f)",
                  best$max_depth, best$eta, best$nrounds, best_auc))
  best
}

# Convert response factor to 0/1 for xgb.cv
label_01 <- as.integer(cv_data$response_fct == "Responder")

message("Tuning XGBoost hyperparameters via xgb.cv (not part of evaluation)...")
message("  combined (CIDER + TMB):")
xgb_params_combined <- tune_xgb_params(cv_data[, c("cider_log2", "tmb_log2")], label_01)

message("  CIDER only:")
xgb_params_cider <- tune_xgb_params(cv_data[, "cider_log2", drop = FALSE], label_01)

message("  TMB only:")
xgb_params_tmb <- tune_xgb_params(cv_data[, "tmb_log2", drop = FALSE], label_01)

message("  full (all biomarkers + cohort):")
xgb_params_full <- tune_xgb_params(
  cv_data[, c("cider_log2", "tmb_log2", "msi_high", "hla_loh",
              "driver_POLE", "driver_POLD1", "driver_KEAP1", "driver_STK11",
              "driver_EGFR", "driver_JAK1", "driver_JAK2", "driver_B2M",
              cohort_cols)], label_01)



In [ ]:
# ── Evaluation CV control — identical to 9_roc.ipynb ─────────────────────────
# Single-row tuneGrid per model: no hyperparameter selection occurs within folds.
# OOF predictions from savePredictions='final' are purely evaluative.
set.seed(6222020)

cv_ctrl <- trainControl(
  method          = "repeatedcv",
  number          = 10,
  repeats         = 10,
  classProbs      = TRUE,
  summaryFunction = twoClassSummary,
  savePredictions = "final"
)

# ── XGBoost models ────────────────────────────────────────────────────────────
message("Fitting XGBoost models with fixed hyperparameters (evaluative CV only)...")

xgb_combined <- train(
  response_fct ~ cider_log2 + tmb_log2,
  data      = cv_data,
  method    = "xgbTree",
  metric    = "ROC",
  trControl = cv_ctrl,
  tuneGrid  = xgb_params_combined,
  verbose   = FALSE
)

xgb_cider <- train(
  response_fct ~ cider_log2,
  data      = cv_data,
  method    = "xgbTree",
  metric    = "ROC",
  trControl = cv_ctrl,
  tuneGrid  = xgb_params_cider,
  verbose   = FALSE
)

xgb_tmb <- train(
  response_fct ~ tmb_log2,
  data      = cv_data,
  method    = "xgbTree",
  metric    = "ROC",
  trControl = cv_ctrl,
  tuneGrid  = xgb_params_tmb,
  verbose   = FALSE
)

xgb_full_features <- c("cider_log2", "tmb_log2", "msi_high", "hla_loh",
                        "driver_POLE", "driver_POLD1", "driver_KEAP1", "driver_STK11",
                        "driver_EGFR", "driver_JAK1", "driver_JAK2", "driver_B2M",
                        cohort_cols)
xgb_full <- train(
  x         = as.matrix(cv_data[, xgb_full_features]),
  y         = cv_data$response_fct,
  method    = "xgbTree",
  metric    = "ROC",
  trControl = cv_ctrl,
  tuneGrid  = xgb_params_full,
  verbose   = FALSE
)


In [ ]:
# ── Logistic regression models (reproduced from 9_roc.ipynb for direct comparison) ──
message("Fitting logistic regression models...")

lr_combined <- train(
  response_fct ~ cider_log2 + tmb_log2,
  data      = cv_data,
  method    = "glm",
  family    = "binomial",
  metric    = "ROC",
  trControl = cv_ctrl
)

lr_cider <- train(
  response_fct ~ cider_log2,
  data      = cv_data,
  method    = "glm",
  family    = "binomial",
  metric    = "ROC",
  trControl = cv_ctrl
)

lr_tmb <- train(
  response_fct ~ tmb_log2,
  data      = cv_data,
  method    = "glm",
  family    = "binomial",
  metric    = "ROC",
  trControl = cv_ctrl
)

# Ridge logistic regression for multi-biomarker models.
# Lambda pre-tuned via cv.glmnet (alpha=0), then fixed for evaluative CV —
# same two-step philosophy as XGBoost: tune separately, evaluate cleanly.
library(glmnet)

# Full model: all biomarkers + cohort (factor handled natively by glm/glmnet)
X_full <- model.matrix(
  ~ cider_log2 + tmb_log2 + msi_high + hla_loh +
    driver_POLE + driver_POLD1 + driver_KEAP1 + driver_STK11 +
    driver_EGFR + driver_JAK1 + driver_JAK2 + driver_B2M + cohort_broad,
  cv_data
)[, -1]
y_bin <- as.integer(cv_data$response_fct == "Responder")
set.seed(6222020)
lambda_full <- glmnet::cv.glmnet(X_full, y_bin, alpha = 0,
  family = "binomial", nfolds = 5)$lambda.min

lr_full <- train(
  x         = X_full,
  y         = cv_data$response_fct,
  method    = "glmnet",
  metric    = "ROC",
  trControl = cv_ctrl,
  tuneGrid  = data.frame(alpha = 0, lambda = lambda_full)
)


In [73]:
#lr_full_ridge

# 2 - Extract and compare CV AUCs

In [86]:
# Extract CV AUC for each model.
# With a single-row tuneGrid, fit$results has exactly one row.
get_cv_auc <- function(fit, model_name, method_name) {
  r <- fit$results
  tibble::tibble(
    model    = model_name,
    method   = method_name,
    auc_mean = signif(r$ROC,   3),
    auc_sd   = round(r$ROCSD, 3)
  )
}

cv_results <- dplyr::bind_rows(
  get_cv_auc(xgb_full, "Full", "XGBoost"),
  get_cv_auc(xgb_combined, "CIDER + TMB", "XGBoost"),
  get_cv_auc(xgb_cider,    "CIDER Score", "XGBoost"),
  get_cv_auc(xgb_tmb,      "TMB",         "XGBoost"),
  get_cv_auc(lr_full,  "Full", "Logistic"),
  get_cv_auc(lr_combined,  "CIDER + TMB", "Logistic"),
  get_cv_auc(lr_cider,     "CIDER Score", "Logistic"),
  get_cv_auc(lr_tmb,       "TMB",         "Logistic")
)

message("CV AUC (10-fold x 10 repeats):")
print(cv_results)

CV AUC (10-fold x 10 repeats):



# A tibble: 9 × 4
  model       method   auc_mean auc_sd
  <chr>       <chr>       <dbl>  <dbl>
1 Full        XGBoost     0.706  0.068
2 CIDER + TMB XGBoost     0.715  0.065
3 CIDER Score XGBoost     0.62   0.075
4 TMB         XGBoost     0.664  0.057
5 Full        Logistic    0.726  0.056
6 Bigger      Logistic    0.725  0.064
7 CIDER + TMB Logistic    0.734  0.064
8 CIDER Score Logistic    0.652  0.077
9 TMB         Logistic    0.682  0.066


# 2b - Cohort-stratified AUC from pan-cancer OOF predictions

Subsets the existing CV out-of-fold predictions by `cohort_broad` — no refitting.
`rowIndex` maps each OOF prediction back to its row in `cv_data`.

In [ ]:
# Compute per-cohort and pan-cancer AUC with 95% DeLong CI from OOF predictions.
# rowIndex is 1-based and maps directly to cv_data rows.
get_cohort_auc <- function(fit, model_name, method_name) {
  preds <- dplyr::inner_join(
    fit$pred, fit$bestTune,
    by = intersect(names(fit$bestTune), names(fit$pred))
  )
  preds$cohort_broad    <- as.character(cv_data$cohort_broad[preds$rowIndex])
  preds$cohort_broad_go <- as.character(cv_data$cohort_broad_go[preds$rowIndex])

  compute_auc_ci <- function(df) {
    tryCatch({
      roc_obj <- pROC::roc(df$obs, df$Responder,
        levels = c('NonResponder', 'Responder'), quiet = TRUE)
      ci <- as.numeric(pROC::ci.auc(roc_obj, conf.level = 0.95, method = 'delong'))
      tibble::tibble(
        auc        = ci[2],
        auc_lower  = ci[1],
        auc_upper  = ci[3],
        n_patients = dplyr::n_distinct(df$rowIndex)
      )
    }, error = function(e) {
      tibble::tibble(auc = NA_real_, auc_lower = NA_real_, auc_upper = NA_real_, n_patients = 0L)
    })
  }

  # Per-cohort AUC with DeLong CI
  cohort_res <- preds |>
    dplyr::group_by(cohort_broad, cohort_broad_go) |>
    dplyr::group_modify(~ compute_auc_ci(.x)) |>
    dplyr::ungroup()

  # Pan-cancer: all OOF predictions combined
  pan_res <- compute_auc_ci(preds) |>
    dplyr::mutate(cohort_broad = 'Pan-Cancer', cohort_broad_go = 'Pan-Cancer')

  dplyr::bind_rows(pan_res, cohort_res) |>
    dplyr::mutate(
      model = model_name, method = method_name,
      cohort_broad_go = dplyr::if_else(
        cohort_broad == "Pan-Cancer",
        paste0("ICI-Treated (N = ", n_patients, ")"),
        cohort_broad_go
      )
    )
}

cohort_results <- dplyr::bind_rows(
  get_cohort_auc(xgb_full,     'Full',        'XGBoost'),
  get_cohort_auc(xgb_combined, 'CIDER + TMB', 'XGBoost'),
  get_cohort_auc(xgb_cider,    'CIDER Score', 'XGBoost'),
  get_cohort_auc(xgb_tmb,      'TMB',         'XGBoost'),
  get_cohort_auc(lr_full,      'Full',        'Ridge LR'),
  get_cohort_auc(lr_combined,  'CIDER + TMB', 'Logistic'),
  get_cohort_auc(lr_cider,     'CIDER Score', 'Logistic'),
  get_cohort_auc(lr_tmb,       'TMB',         'Logistic')
) |>
  dplyr::mutate(
    model  = factor(model,  levels = c('CIDER Score', 'TMB', 'CIDER + TMB', 'Full')),
    method = factor(method, levels = c('Logistic', 'Ridge LR', 'XGBoost')),
    cohort_label = paste0(cohort_broad, ' (N=', n_patients, ')')
  )

# ── Plot: AUC per cohort, faceted by cohort, coloured by model ───────────────
p_cohort_auc <- cohort_results |>
  ggplot2::ggplot(ggplot2::aes(
    x = auc, y = model, colour = model, shape = method, group = model
  )) +
  ggplot2::geom_vline(xintercept = 0.5, linetype = 'dashed', colour = COL_NEGATIVE) +
  ggplot2::geom_line(colour = COL_NEGATIVE, linewidth = 0.6) +
  ggplot2::geom_point(size = 3) +
  ggplot2::facet_wrap(~ cohort_label, nrow = 1) +
  # 'CIDER + TMB' gets COL_POSITIVE (orange) -- the colour reserved
  # project-wide for the headline combined model in ROC/AUC
  # model-comparison panels (see helpers/themes.r); 'CIDER Score' alone
  # gets COL_ACCENT (blue) instead, matching 9_roc.ipynb's roc_palette and
  # this notebook's own MODEL_COLORS/model_colors below.
  ggplot2::scale_colour_manual(
    values = c('CIDER Score' = COL_ACCENT, 'TMB' = COL_NEGATIVE,
               'CIDER + TMB' = COL_POSITIVE, 'Full' = COL_MID),
    guide = 'none'
  ) +
  ggplot2::scale_shape_manual(
    values = c('Logistic' = 16, 'Ridge LR' = 15, 'XGBoost' = 17),
    name = 'Model'
  ) +
  ggplot2::scale_x_continuous(limits = c(0.3, 1), breaks = c(0.5, 0.75, 1.0)) +
  go_theme +
  ggplot2::theme(panel.grid.major.y = ggplot2::element_line(colour = COL_NEGATIVE)) +
  ggplot2::labs(
    x = 'AUC (CV out-of-fold)', y = NULL,
    title = 'Cohort-Stratified AUC'
  )

set_plot_dims(14, 4); p_cohort_auc
print(cohort_results |> dplyr::select(cohort_broad, model, method, auc, n_patients))

# 3 - Build ROC curves from CV held-out predictions

In [12]:
# ROC curves built from CV out-of-fold predictions.
#
# Filtering fix: use dplyr::inner_join on bestTune rather than == on numeric
# columns, which is unreliable due to floating point comparison.
# A diagnostic check confirms the row count is exactly n_samples * n_repeats.
get_cv_roc_curve <- function(fit, model_name, method_name) {
  preds <- fit$pred

  # Filter to best-tuned hyperparameters via join (safe for numeric params)
  preds <- dplyr::inner_join(preds, fit$bestTune,
                              by = intersect(names(fit$bestTune), names(preds)))

  # Sanity check: each sample should appear once per repeat
  expected_rows <- nrow(cv_data) * 10L  # 10 repeats
  if (nrow(preds) != expected_rows) {
    warning(model_name, " (", method_name, "): expected ", expected_rows,
            " OOF predictions, got ", nrow(preds),
            " — ROC curve may be unreliable.")
  }

  roc_obj <- pROC::roc(preds$obs, preds$Responder,
                        levels = c("NonResponder", "Responder"), quiet = TRUE)
  coords  <- pROC::coords(roc_obj, "all",
                           ret = c("threshold", "sensitivity", "specificity"),
                           transpose = FALSE)
  dplyr::mutate(coords,
    model  = model_name,
    method = method_name,
    auc    = round(as.numeric(pROC::auc(roc_obj)), 3)
  )
}

roc_curves <- dplyr::bind_rows(
  get_cv_roc_curve(xgb_full,     "Full",        "XGBoost"),
  get_cv_roc_curve(xgb_combined, "CIDER + TMB", "XGBoost"),
  get_cv_roc_curve(xgb_cider,    "CIDER Score", "XGBoost"),
  get_cv_roc_curve(xgb_tmb,      "TMB",         "XGBoost"),
  get_cv_roc_curve(lr_full,       "Full",        "Logistic"),
  get_cv_roc_curve(lr_combined,  "CIDER + TMB", "Logistic"),
  get_cv_roc_curve(lr_cider,     "CIDER Score", "Logistic"),
  get_cv_roc_curve(lr_tmb,       "TMB",         "Logistic")
)

# Save lr_full OOF ROC curve for overlay in 9_roc.ipynb overall_roc_plot
lr_full_summary  <- get_cv_auc(lr_full, "Full (Regularized)", "Logistic")
lr_full_roc_data <- get_cv_roc_curve(lr_full, "Full (Regularized)", "Logistic") |>
  dplyr::rename(predictor = model) |>
  dplyr::mutate(auc_mean = lr_full_summary$auc_mean, auc_sd = lr_full_summary$auc_sd)
saveRDS(lr_full_roc_data, paste0(CIDER_PROJ_DIR, "figs/lr_full_roc_curve.rds"))
message("Saved lr_full_roc_curve.rds")


In [ ]:
# ── Combined pan-cancer + cohort AUC barplot ──────────────────────────────────
# Pan-cancer AUC (mean ± SD from 10×10 CV) + per-cohort AUC from OOF subsets.
# Bars grouped by method (LR vs XGBoost); faceted by cohort.

MODEL_LEVELS  <- c('CIDER Score', 'TMB', 'CIDER + TMB', 'Full')
# 'CIDER + TMB' is the headline combined model (same one 9_roc.ipynb's
# Figure 5B highlights as "TMB + CIDER Score"), so it gets COL_POSITIVE
# (orange) -- the colour reserved project-wide for the headline "CIDER +
# TMB" model in ROC/AUC model-comparison panels (see helpers/themes.r).
# 'CIDER Score' alone takes COL_ACCENT (blue) instead -- swapped from this
# palette's otherwise fixed POSITIVE/NEGATIVE/MID/ACCENT order so the two
# never collide; 'Full' (the larger multi-feature model) keeps the sage
# green slot. Matches 9_roc.ipynb's roc_palette exactly.
MODEL_COLORS <- c(
  'CIDER Score' = COL_ACCENT,
  'TMB'         = COL_NEGATIVE,
  'CIDER + TMB' = COL_POSITIVE,
  'Full'        = COL_MID
)

# ── All cohorts + Pan-cancer: DeLong 95% CI from OOF predictions ─────────────
# Pan-cancer row included automatically by get_cohort_auc (all OOF combined).
cohort_df <- dplyr::bind_rows(
  get_cohort_auc(lr_full,      'Full',        'Logistic'),
  get_cohort_auc(lr_combined,  'CIDER + TMB', 'Logistic'),
  get_cohort_auc(lr_cider,     'CIDER Score', 'Logistic'),
  get_cohort_auc(lr_tmb,       'TMB',         'Logistic')
)

# ── Combine and factor-order cohorts ─────────────────────────────────────────
# cohort_broad_go here is THIS notebook's own merged-cohort label (ICI-MSI /
# Other MSS, computed in the data-prep cell above), not Figure 3's -- the
# cohort definitions differ (Figure 3 keeps Non-CRC MSI and CRC-MSI as
# separate cohorts; this figure pools them), so the two aren't the same
# label set despite the similar "(N = ...)" format.
pan_cancer_label <- cohort_df |>
  dplyr::filter(cohort_broad == 'Pan-Cancer') |>
  dplyr::pull(cohort_broad_go) |>
  unique()

# cohort_broad now carries this figure's own merged scheme (see the
# ICI-MSI/Other MSS merge in the data-prep cell above) -- SKCM/NSCLC/BLCA in
# their usual order, then Other MSS, then the pooled ICI-MSI cohort last
# (same "MSI cohort(s) go last" convention BROAD_LEVELS/
# BROAD_LEVELS_OTHER_SPLIT use elsewhere, just with Non-CRC MSI and CRC-MSI
# already pooled into one entry instead of two). BROAD_LEVELS itself isn't
# used here since it doesn't know about either merged cohort name.
NON_PAN_COHORT_ORDER <- c("SKCM", "NSCLC", "BLCA", "Other MSS", "ICI-MSI")

cohort_go_order <- c(
  pan_cancer_label,
  cohort_df |>
    dplyr::filter(cohort_broad != 'Pan-Cancer') |>
    dplyr::distinct(cohort_broad, cohort_broad_go) |>
    dplyr::mutate(cohort_broad = factor(cohort_broad, levels = NON_PAN_COHORT_ORDER)) |>
    dplyr::arrange(cohort_broad) |>
    dplyr::pull(cohort_broad_go)
)

all_auc <- cohort_df |>
  dplyr::select(cohort_broad_go, model, method, auc, auc_lower, auc_upper) |>
  dplyr::mutate(
    model           = factor(model, levels = MODEL_LEVELS),
    cohort_broad_go = factor(as.character(cohort_broad_go), levels = cohort_go_order),
    auc       = auc,
    auc_lower = auc_lower,
    auc_upper = auc_upper
  )

# Diagnostic: show any cohort_broad_go values that became NA
na_cohorts <- cohort_df$cohort_broad_go[
  !cohort_df$cohort_broad_go %in% cohort_go_order]
if (length(na_cohorts) > 0)
  message("cohort_broad_go values not in factor levels: ",
          paste(unique(na_cohorts), collapse = ", "))

# ── Plot ──────────────────────────────────────────────────────────────────────
p_bar <- all_auc |>
  dplyr::filter(!is.na(cohort_broad_go)) |>
  ggplot2::ggplot(ggplot2::aes(
    x = model, y = auc, fill = model
  )) +
  ggplot2::geom_hline(yintercept = 0.5, linetype = 'dashed', colour = COL_NEGATIVE) +
  ggplot2::geom_col(
    width = 0.7, alpha = 0.6, colour = COL_INK, linewidth = 0.3
  ) +
  ggplot2::geom_text(
    ggplot2::aes(label = sprintf('%.2f', auc), y = auc),
    vjust = -0.4, size = 3.5, colour = COL_INK
  ) +

  ggplot2::facet_wrap(~ cohort_broad_go, nrow = 2, ncol = 3) +
  ggplot2::scale_fill_manual(values = MODEL_COLORS, name = NULL, guide = 'none') +
  ggplot2::scale_y_continuous(
    limits = c(0, 1), breaks = c(0, 0.25, 0.5, 0.75, 1.0),
    expand = ggplot2::expansion(mult = c(0, 0.05))
  ) +
  ggplot2::scale_x_discrete(guide = ggplot2::guide_axis(n.dodge = 2)) +
  go_theme +
  ggplot2::theme(
    strip.text = ggplot2::element_text(face = 'bold', size = 9)
  ) +
  ggplot2::labs(
    x     = 'Model',
    y     = 'AUC',
    title = 'Logistic Regression — Pan-Cancer and Cohort-Level AUC'
  )

set_plot_dims(8, 6); p_bar

ggsave(paste0(CIDER_PROJ_DIR, 'figs/supplement_xgboost_cohort_auc.png'),
       p_bar, width = 8, height = 6, dpi = 300)
message('Saved supplement_xgboost_cohort_auc.png')

# 4 - Figures

In [ ]:
# Same cycle as MODEL_COLORS above / 9_roc.ipynb -- "CIDER + TMB" (the
# headline combined model) gets COL_POSITIVE (orange), "CIDER Score" alone
# gets COL_ACCENT (blue), "Full" (the larger multi-feature model) keeps the
# sage green slot.
model_colors <- c(
  "CIDER + TMB" = COL_POSITIVE,
  "CIDER Score" = COL_ACCENT,
  "TMB"         = COL_NEGATIVE,
  "Full"        = COL_MID
)
method_lty <- c("Logistic" = "solid", "XGBoost" = "dashed")

# ── Panel A: ROC curves ───────────────────────────────────────────────────────
p_roc <- roc_curves |>
  dplyr::mutate(
    model      = factor(model,  levels = c("Full", "CIDER + TMB", "CIDER Score", "TMB")),
    method     = factor(method, levels = c("Logistic", "XGBoost")),
    line_alpha = dplyr::if_else(model == "CIDER + TMB", 1.0, 0.4),
    line_width = dplyr::if_else(model == "CIDER + TMB" & method == "Logistic", 1.3, 0.8)
  ) |>
  ggplot(aes(x = 1 - specificity, y = sensitivity,
             colour = model, linetype = method,
             alpha = line_alpha, linewidth = line_width,
             group = interaction(model, method))) +
  geom_line() +
  geom_abline(slope = 1, intercept = 0, linetype = "dotted", colour = COL_NEGATIVE) +
  scale_colour_manual(values = model_colors, name = "Predictor") +
  scale_linetype_manual(values = method_lty, name = "Model") +
  scale_alpha_identity() +
  scale_linewidth_identity() +
  scale_x_continuous(labels = scales::percent_format(), limits = c(0, 1),
                     expand = expansion(mult = c(0, 0.01))) +
  scale_y_continuous(labels = scales::percent_format(), limits = c(0, 1),
                     expand = expansion(mult = c(0, 0.01))) +
  go_theme +
  theme(
    legend.position  = "bottom",
    legend.direction = "horizontal",
    legend.box       = "horizontal"
  ) +
  guides(
    colour   = guide_legend(nrow = 2, title.position = "top"),
    linetype = guide_legend(nrow = 2, title.position = "top")
  ) +
  labs(
    title = "ROC Curves (CV Out-of-Fold Predictions)",
    x     = "1 − Specificity (FPR)",
    y     = "Sensitivity (TPR)"
  )
set_plot_dims(7, 5); p_roc

In [ ]:
# ── Panel B: CV AUC comparison ────────────────────────────────────────────────
p_auc <- cv_results |>
  dplyr::mutate(
    model  = factor(model,  levels = c("TMB", "CIDER Score", "CIDER + TMB", "Full")),
    method = factor(method, levels = c("Logistic", "XGBoost"))
  ) |>
  ggplot(aes(x = auc_mean, y = model, colour = model, shape = method,
             group = model)) +
  # geom_line connects each model's Logistic/XGBoost estimates (the "CV and
  # MOEs" -- cross-validated AUC and its margin-of-error whiskers); it is
  # already scoped to just those two points per model via aes(group = model),
  # so no gridline overlay is needed to visually tie them together.
  geom_line(colour = COL_NEGATIVE, linewidth = 0.8) +
  geom_errorbarh(aes(xmin = auc_mean - auc_sd, xmax = auc_mean + auc_sd),
                 height = 0.15, linewidth = 0.6, alpha = 0.6) +
  geom_point(size = 3.5, alpha = 0.6) +
  scale_colour_manual(values = model_colors, guide = "none") +
  scale_shape_manual(values = c("Logistic" = 16, "XGBoost" = 17), name = "Model") +
  scale_x_continuous(limits = c(0.45, NA), n.breaks = 5) +
  go_theme +
  # (panel.grid.major.y removed -- it drew a full-width horizontal track
  # behind every model row, which read as the CV/MOE connector extending
  # across the whole plot. go_theme already blanks major gridlines by
  # default, so the real per-model geom_line() above is now the only
  # visible connector, matching the fix applied to 10_unique_vs_total.ipynb.)
  theme(
    legend.position    = "bottom",
    legend.direction   = "horizontal"
  ) +
  guides(shape = guide_legend(nrow = 1, title.position = "top")) +
  labs(
    x     = "CV AUC (mean \u00b1 SD, 10-fold \u00d7 10 repeats)",
    y     = NULL,
    title = "Cross-Validated AUC: XGBoost vs Logistic"
  )

set_plot_dims(6, 3.5); p_auc

In [ ]:
# ── Assemble and save individual ROC/AUC figure ──────────────────────────────
# Keep individual save for reference; combined figure assembled below.
supplement_xgboost <-
  (p_roc | p_auc)

saveRDS(supplement_xgboost,
        paste0(CIDER_PROJ_DIR, "figs/supplement_xgboost_roc.rds"))
ggsave(paste0(CIDER_PROJ_DIR, "figs/supplement_xgboost_roc.png"),
       supplement_xgboost + figure_annotation(
         "XGBoost vs Logistic Regression \u2014 CIDER T-cell Score and TMB"),
       width = 13, height = 5.5, dpi = 300)
message("Saved supplement_xgboost_roc.png")

# ── Combined figure: cohort AUC (A) / ROC comparison (B) ─────────────────────
combined_xgb <- (p_bar / supplement_xgboost) +
  patchwork::plot_annotation(
    title    = "A Simple CIDER + TMB Model Achieves Competitive Predictive Performance",
    subtitle = "XGBoost vs Logistic Regression \u2014 CIDER T-cell Score and TMB",
    caption  = "Full model features: CIDER T-cell Score, TMB, MSI status, HLA LOH, driver gene mutations (POLE, POLD1, KEAP1, STK11, EGFR, JAK1, JAK2, B2M), and tumour type indicators.",
    tag_levels = "A",
    theme = go_theme + ggplot2::theme(
      plot.title    = ggplot2::element_text(hjust = 0.5, size = 13, face = "bold"),
      plot.subtitle = ggplot2::element_text(hjust = 0.5, size = 10),
      plot.caption  = ggplot2::element_text(hjust = 0, size = 7.5,
                        colour = COL_NEGATIVE, margin = ggplot2::margin(t = 6))
    )
  )

set_plot_dims(13 * 0.85, 12 * 0.85); combined_xgb

ggsave(paste0(CIDER_PROJ_DIR, "figs/supplement_xgboost_combined.png"),
       combined_xgb, width = 10 * 0.85, height = 12 * 0.85, dpi = 300)
message("Saved supplement_xgboost_combined.png")
